In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from delta.tables import DeltaTable

# Widget with default value
dbutils.widgets.text("table_name", "dim_date")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
gold_base = "abfss://curated@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Date series
start_date = "2016-01-01"
end_date = "2018-12-31"

df_dates = (spark.createDataFrame([(start_date, end_date)], ["start", "end"])
            .select(F.explode(F.sequence(F.to_date("start"), F.to_date("end"))).alias("d")))

# Table Logic
df_dim_date = (df_dates
                .select(
                    F.date_format(F.col("d"), "yyyyMMdd").cast("int").alias("date_key"),
                    F.col("d").alias("date"),
                    F.dayofmonth(F.col("d")).alias("day"),
                    F.weekofyear(F.col("d")).alias("week"),
                    F.month(F.col("d")).alias("month"),
                    F.date_format(F.col("d"), "MMMM").alias("month_name"),
                    F.quarter(F.col("d")).alias("quarter"),
                    F.year(F.col("d")).alias("year"),
                    F.when(F.dayofweek(F.col("d")).isin(1,7), 1).otherwise(0).alias("is_weekend"),
                    F.when(
                        ((F.month(F.col("d")) == 1) & (F.dayofmonth(F.col("d")) == 1)) | # New Year
                        ((F.month(F.col("d")) == 4) & (F.dayofmonth(F.col("d")) == 21)) | # Tiradentes
                        ((F.month(F.col("d")) == 5) & (F.dayofmonth(F.col("d")) == 1)) | # Labor Day
                        ((F.month(F.col("d")) == 9) & (F.dayofmonth(F.col("d")) == 7)) | # Independence
                        ((F.month(F.col("d")) == 10) & (F.dayofmonth(F.col("d")) == 12)) | # Our Lady of Aparecida
                        ((F.month(F.col("d")) == 11) & (F.dayofmonth(F.col("d")) == 2)) | # All Souls
                        ((F.month(F.col("d")) == 11) & (F.dayofmonth(F.col("d")) == 15)) | # Republic Proclamation
                        ((F.month(F.col("d")) == 12) & (F.dayofmonth(F.col("d")) == 25)), # Christmas
                        1
                    ).otherwise(0).alias("is_holiday_brazil_flag"),
                    F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
                ))

# Unknown record
now = datetime.now().replace(microsecond=0)
unknown_data = [(-1, datetime(1999, 1, 1).date(), 0, 0, 0, "N/A", 0, 0, 0, 0, now)]
target_schema = df_dim_date.schema
df_unknown = spark.createDataFrame(unknown_data, schema=target_schema)
df_final = df_dim_date.unionByName(df_unknown)

if DeltaTable.isDeltaTable(spark, gold_path):
    print(f"--> [Merge] Updating {target_table} to prevent duplicates.")
    dt = DeltaTable.forPath(spark, gold_path)

    # Merge Condition
    data_cols = [c for c in df_final.columns if c not in ["date_key", "gold_load_at"]]
    change_condition = " OR ".join([f"NOT (target.{c} <=> source.{c})" for c in data_cols])

    (dt.alias("target")
     .merge(df_final.alias("source"), "target.date_key = source.date_key")
     .whenMatchedUpdateAll(condition = change_condition)
     .whenNotMatchedInsertAll()
     .execute())
else:
    print(f"--> [INIT] Creating {target_table} for the first time.")
    df_final.write.format("delta").save(gold_path)

    # Z-Ordering on Year and Month for faster time-series queries
    spark.sql(f"OPTIMIZE delta.`{gold_path}` ZORDER BY (year, month)")


# 3. Audit History
dt_final = DeltaTable.forPath(spark, gold_path)
last_op = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_ins = int(last_op.get("numTargetRowsInserted", 0))
rows_upd = int(last_op.get("numTargetRowsUpdated", 0))

message = f"--> Success | Inserted: {rows_ins} | Updated: {rows_upd}"
print(message)
dbutils.notebook.exit(message)